# Raw books: first EDA

Explore the saved catalog to understand its structure, missing information, repeated identifiers, and distributions.

**Scope:** descriptive inspection of `data/raw/books.parquet`. Findings describe this artifact; it was ordered by `parent_asin`, not randomly sampled. A row is provisionally an Amazon parent-book record, not necessarily a distinct work.

Run with the project's **.venv** Python kernel in VS Code. Use **Restart Kernel → Run All** so every result is reproducible. The notebook works from the repository root or its notebooks directory. A missing Parquet file raises an error.

**Stopping point:** no row removal, missing-value imputation, feature selection, scaling, resampling, chunking, embeddings, model fitting, or evaluation. Numeric conversion below is an in-memory diagnostic copy. A final cell explains the evaluation decisions needed before continuing.

Objective: Load the libraries and show which Python environment and versions we are using.

In [1]:
import hashlib
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from bookpath.local_data import read_books_from_parquet

print(f"Python: {sys.executable}")
print(f"pandas: {pd.__version__}")
print(f"Python version: {sys.version}")


Python: /Users/dominiclizarraga/code/dominiclizarraga/bookpath_v2/.venv/bin/python
pandas: 3.0.5
Python version: 3.12.9 (main, Mar 17 2025, 21:36:21) [Clang 20.1.0 ]


## 1. Identify and read the local artifact

A checksum is a fingerprint calculated from the file's bytes. We store `checksum_before` before reading the data, then calculate `checksum_after` at the end and compare them. This checks the saved Parquet file; it does not check every value in the in-memory DataFrame.

Objective: Read the local Parquet file, show its size and shape, and record its checksum before EDA.

In [2]:
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "pyproject.toml").is_file() and (candidate / "src/bookpath").is_dir():
        project_root = candidate
        break
else:
    raise FileNotFoundError("Open this notebook from inside the Bookpath repository.")

raw_path = project_root / "data/raw/books.parquet"


def file_checksum(path):
    with path.open("rb") as raw_file:
        return hashlib.file_digest(raw_file, "sha256").hexdigest()


checksum_before = file_checksum(raw_path)
books = read_books_from_parquet(raw_path)
row_count = len(books)
assert row_count > 0, "The saved dataset is empty."

print(f"Rows: {row_count:,}; columns: {len(books.columns)}")
print(f"File size: {raw_path.stat().st_size:,} bytes")
print(f"SHA-256: {checksum_before}")

Rows: 9,034; columns: 38
File size: 92,373,274 bytes
SHA-256: 4acf6979b15215c28f2a45a1488ace4d749e2573f0fa4a0110e2933dd0194dd8


## 2. Inspect the schema

`str` means a string column. `object` can contain collections such as arrays of TOC entries. A numeric-looking string still needs an explicit conversion before numeric analysis.

Objective: Show each column's name, data type, and number of null values.

In [3]:
schema = pd.DataFrame({
    "column_name": books.columns,
    "dtype": books.dtypes.astype(str).values,
    "null_rows": books.isna().sum().values,
})

schema.index = range(1, len(schema) + 1)

display(schema)

,column_name,dtype,null_rows
1,amazon_business_money_record_number,str,0
2,amazon_isbn_10,str,0
3,amazon_isbn_13,str,0
4,amazon_raw_record_json,str,0
5,author_json,str,0
6,average_rating,str,0
7,bought_together_json,str,0
8,canonical_isbn_13,str,0
9,categories,object,0
10,description,object,0


In [4]:
books["amazon_raw_record_json"].iloc[0]

'{"main_category": "Books", "title": "The wheels of commerce, vol.2: civilisation and capitalism 15th-18th", "subtitle": "Hardcover \\u2013 Import, January 1, 1982", "author": null, "average_rating": 4.9, "rating_number": 13, "features": [], "description": [], "price": 12.99, "images": [], "videos": [], "store": "Fernand Braudel (Author),  Sian Reynolds (Translator)", "categories": ["Books", "Business & Money", "Economics"], "details": {"Publisher": "Harper Collins; First Edition (January 1, 1982)", "Language": "English", "Hardcover": "670 pages", "ISBN 10": "000216132X", "ISBN 13": "978-0002161329", "Item Weight": "0.035 ounces"}, "parent_asin": "000216132X", "bought_together": null}'

In [5]:
books.head(2)

,amazon_business_money_record_number,amazon_isbn_10,amazon_isbn_13,amazon_raw_record_json,author_json,average_rating,bought_together_json,canonical_isbn_13,categories,description,...,rating_number,raw_toc_item_count,raw_toc_json,raw_toc_shape,store,subtitle,title,toc_entries,toc_entry_count,videos_json
0,100531,000216132X,9780002161329,"{""main_category"": ""Books"", ""title"": ""The wheel...",null,4.9,null,9780002161329,"[Books, Business & Money, Economics]",[],...,13,3,"[{""type"":""/type/text"",""value"":""v. 1. The struc...",list[dict],"Fernand Braudel (Author), Sian Reynolds (Tran...","Hardcover – Import, January 1, 1982","The wheels of commerce, vol.2: civilisation an...","[{'level': None, 'page': None, 'sequence': '1'...",3,[]
1,97051,0007519532,9780007519538,"{""main_category"": ""Books"", ""title"": ""Will ther...","{""avatar"":""https://m.media-amazon.com/images/S...",3.9,null,9780007519538,"[Books, Business & Money, Management & Leaders...","[Review, ‘It's relevant, useful and fun to rea...",...,36,5,"[{""level"":0,""title"":""1. Nearly meeting"",""type""...",list[dict],David Pearl (Author),"Paperback – October 1, 2013",Will there be Donuts?: Better Business One Mee...,"[{'level': '0', 'page': None, 'sequence': '1',...",5,[]


### Inspect one random record

The `amazon_raw_record_json` column preserves the original Amazon record as a JSON string. The DataFrame row contains extracted Amazon fields plus additional OpenLibrary and TOC fields, so the two are not expected to have identical columns. Each run selects a new random record. We compare its overlapping Amazon fields after decoding JSON and normalizing arrays and numbers. One record is useful for understanding the structure; it does not prove that every record matches.

Objective: Select a new random row on each run, display its fields A–Z with numbered rows, and compare them with its source JSON.

In [6]:
random_generator = np.random.default_rng()
random_position = int(random_generator.integers(0, len(books)))
random_row = books.iloc[random_position]
amazon_record = json.loads(random_row["amazon_raw_record_json"])

dataframe_view = (
    random_row.rename_axis("column_name")
    .reset_index(name="dataframe_value")
    .sort_values("column_name")
    .reset_index(drop=True)
)
dataframe_view.index = range(1, len(dataframe_view) + 1)
dataframe_view.index.name = "number"

amazon_json_view = (
    pd.Series(amazon_record, name="amazon_json_value")
    .rename_axis("json_key")
    .reset_index()
    .sort_values("json_key")
    .reset_index(drop=True)
)
amazon_json_view.index = range(1, len(amazon_json_view) + 1)
amazon_json_view.index.name = "number"

print(f"Random row position: {random_position}")
print(f"DataFrame index label: {random_row.name}")
print(f"DataFrame fields: {len(dataframe_view)}")
display(dataframe_view)
print(f"Amazon JSON fields: {len(amazon_json_view)}")
display(amazon_json_view)

json_to_dataframe_column = {
    "main_category": "main_category",
    "title": "title",
    "subtitle": "subtitle",
    "author": "author_json",
    "average_rating": "average_rating",
    "rating_number": "rating_number",
    "features": "features",
    "description": "description",
    "price": "price_json",
    "images": "images_json",
    "videos": "videos_json",
    "store": "store",
    "categories": "categories",
    "details": "details_json",
    "parent_asin": "parent_asin",
    "bought_together": "bought_together_json",
}
numeric_amazon_columns = {"average_rating", "rating_number"}


def decoded_dataframe_value(json_key, dataframe_column):
    value = random_row[dataframe_column]
    if dataframe_column.endswith("_json"):
        return json.loads(value)
    if json_key in numeric_amazon_columns:
        return pd.to_numeric(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value


comparison_rows = []
for json_key, dataframe_column in json_to_dataframe_column.items():
    json_value = amazon_record.get(json_key)
    dataframe_value = decoded_dataframe_value(json_key, dataframe_column)
    comparison_rows.append({
        "json_key": json_key,
        "dataframe_column": dataframe_column,
        "matches_after_decoding": dataframe_value == json_value,
        "amazon_json_value": json_value,
        "dataframe_value": dataframe_value,
    })

comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values("json_key")
    .reset_index(drop=True)
)
comparison.index = range(1, len(comparison) + 1)
comparison.index.name = "number"
display(comparison)
print(f"Matching overlapping fields: {comparison['matches_after_decoding'].sum()} of {len(comparison)}")

Random row position: 6386
DataFrame index label: 6386
DataFrame fields: 38


,column_name,dataframe_value
number,,
1,amazon_business_money_record_number,129159
2,amazon_isbn_10,1451610904
3,amazon_isbn_13,9781451610901
4,amazon_raw_record_json,"{""main_category"": ""Books"", ""title"": ""Risk Inte..."
5,author_json,"{""avatar"":""https://m.media-amazon.com/images/I..."
6,average_rating,3.9
7,bought_together_json,null
8,canonical_isbn_13,9781451610901
9,categories,"[Books, Business & Money, Management & Leaders..."


Amazon JSON fields: 16


,json_key,amazon_json_value
number,,
1,author,{'avatar': 'https://m.media-amazon.com/images/...
2,average_rating,3.9
3,bought_together,None
4,categories,"[Books, Business & Money, Management & Leaders..."
5,description,"[About the Author, Dylan Evans is the founder ..."
6,details,"{'Publisher': 'Free Press (April 17, 2012)', '..."
7,features,[There is a special kind of intelligence for d...
8,images,[]
9,main_category,Books


,json_key,dataframe_column,matches_after_decoding,amazon_json_value,dataframe_value
number,,,,,
1,author,author_json,True,{'avatar': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...
2,average_rating,average_rating,True,3.9,3.9
3,bought_together,bought_together_json,True,None,None
4,categories,categories,True,"[Books, Business & Money, Management & Leaders...","[Books, Business & Money, Management & Leaders..."
5,description,description,True,"[About the Author, Dylan Evans is the founder ...","[About the Author, Dylan Evans is the founder ..."
6,details,details_json,True,"{'Publisher': 'Free Press (April 17, 2012)', '...","{'Publisher': 'Free Press (April 17, 2012)', '..."
7,features,features,True,[There is a special kind of intelligence for d...,[There is a special kind of intelligence for d...
8,images,images_json,True,[],[]
9,main_category,main_category,True,Books,Books


Matching overlapping fields: 16 of 16


A `False` result identifies a field to investigate; it does not automatically mean the DataFrame is wrong because upstream extraction may intentionally normalize a value. A complete validation would run explicit rules across every record.

### Check field presence separately from value equality

This check asks only whether each expected JSON key and DataFrame column exists. A key with JSON `null` is still present. Therefore, `subtitle` can pass this presence check while failing the stricter value comparison when JSON `null` and a pandas missing value use different Python representations.

Objective: Select a new random record and verify that all 16 expected fields are present in both the Amazon JSON structure and the DataFrame schema.

In [14]:
presence_random_generator = np.random.default_rng()
presence_random_position = int(presence_random_generator.integers(0, len(books)))
presence_random_row = books.iloc[presence_random_position]
presence_amazon_record = json.loads(
    presence_random_row["amazon_raw_record_json"]
)

expected_field_mapping = {
    "main_category": "main_category",
    "title": "title",
    "subtitle": "subtitle",
    "author": "author_json",
    "average_rating": "average_rating",
    "rating_number": "rating_number",
    "features": "features",
    "description": "description",
    "price": "price_json",
    "images": "images_json",
    "videos": "videos_json",
    "store": "store",
    "categories": "categories",
    "details": "details_json",
    "parent_asin": "parent_asin",
    "bought_together": "bought_together_json",
}

presence_rows = []
for json_key, dataframe_column in expected_field_mapping.items():
    json_key_present = json_key in presence_amazon_record
    dataframe_column_present = dataframe_column in books.columns
    presence_rows.append({
        "json_key": json_key,
        "dataframe_column": dataframe_column,
        "present_in_amazon_json": json_key_present,
        "present_in_dataframe": dataframe_column_present,
        "present_in_both": json_key_present and dataframe_column_present,
    })

presence_check = (
    pd.DataFrame(presence_rows)
    .sort_values("json_key")
    .reset_index(drop=True)
)
presence_check.index = range(1, len(presence_check) + 1)
presence_check.index.name = "number"

print(f"Random row position: {presence_random_position}")
display(presence_check)
print(
    f"Fields present in both: "
    f"{presence_check['present_in_both'].sum()} of {len(presence_check)}"
)

Random row position: 8136


,json_key,dataframe_column,present_in_amazon_json,present_in_dataframe,present_in_both
number,,,,,
1,author,author_json,True,True,True
2,average_rating,average_rating,True,True,True
3,bought_together,bought_together_json,True,True,True
4,categories,categories,True,True,True
5,description,description,True,True,True
6,details,details_json,True,True,True
7,features,features,True,True,True
8,images,images_json,True,True,True
9,main_category,main_category,True,True,True


Fields present in both: 16 of 16


### Audit title and subtitle across all records

JSON `null` becomes Python `None`, while pandas displays its missing string value as `NaN`. For this source-versus-column audit, those two representations count as the same missing state. Missing values are still counted separately and are not filled or removed.

Objective: Across all 9,034 records, count title and subtitle key presence, non-missing values, equivalent missing pairs, and matching extracted values.

In [15]:
amazon_records = books["amazon_raw_record_json"].map(json.loads)


def values_match_allowing_missing(
    json_key_is_present, json_value, dataframe_value
):
    if not json_key_is_present:
        return False

    json_value_is_missing = bool(pd.isna(json_value))
    dataframe_value_is_missing = bool(pd.isna(dataframe_value))

    if json_value_is_missing or dataframe_value_is_missing:
        return json_value_is_missing and dataframe_value_is_missing
    return json_value == dataframe_value


audit_rows = []
for field in ["title", "subtitle"]:
    json_key_present = amazon_records.map(lambda record: field in record)
    json_values = amazon_records.map(lambda record: record.get(field))
    dataframe_values = books[field]
    json_value_missing = json_values.isna()
    dataframe_value_missing = dataframe_values.isna()
    values_match = pd.Series(
        [
            values_match_allowing_missing(
                json_key_is_present, json_value, dataframe_value
            )
            for json_key_is_present, json_value, dataframe_value in zip(
                json_key_present, json_values, dataframe_values, strict=True
            )
        ],
        index=books.index,
    )

    audit_rows.append({
        "field": field,
        "total_records": len(books),
        "dataframe_column_exists": field in books.columns,
        "json_key_rows": int(json_key_present.sum()),
        "json_key_missing_rows": int((~json_key_present).sum()),
        "json_nonmissing_values": int(
            (json_key_present & ~json_value_missing).sum()
        ),
        "json_null_values": int(
            (json_key_present & json_value_missing).sum()
        ),
        "dataframe_nonmissing_values": int((~dataframe_value_missing).sum()),
        "both_missing_rows": int(
            (
                json_key_present
                & json_value_missing
                & dataframe_value_missing
            ).sum()
        ),
        "matching_values_including_missing": int(values_match.sum()),
        "different_values": int((~values_match).sum()),
    })

title_subtitle_audit = pd.DataFrame(audit_rows).sort_values("field")
title_subtitle_audit.index = range(1, len(title_subtitle_audit) + 1)
title_subtitle_audit.index.name = "number"
display(title_subtitle_audit)

,field,total_records,dataframe_column_exists,json_key_rows,json_key_missing_rows,json_nonmissing_values,json_null_values,dataframe_nonmissing_values,both_missing_rows,matching_values_including_missing,different_values
number,,,,,,,,,,,
1,subtitle,9034,True,9032,2,8106,926,8106,926,9032,2
2,title,9034,True,9034,0,9034,0,9034,0,9034,0


`matching_values_including_missing` answers whether extraction preserved the source value or the same missing state. It does not claim that a missing subtitle should later be imputed, deleted, or used by a model.

## 3. Distinguish nulls, blank text, and empty collections

A null is a missing value. An empty string and an empty list are present values with no content, so `isna()` alone does not find them. All percentages here use the number of raw records as the denominator.

Objective: Find which columns contain null values and show their counts and percentages.

In [20]:
null_counts = books.isna().sum()
null_summary = pd.DataFrame({
    "null_rows": null_counts,
    "percent_of_books": (100 * null_counts / row_count).round(2),
})
display(null_summary.loc[null_counts > 0].sort_values("null_rows", ascending=False))

,null_rows,percent_of_books
ol_subtitle,2088,23.11
subtitle,928,10.27
store,12,0.13
ol_publish_date,3,0.03
main_category,2,0.02


Objective: Count empty or whitespace-only strings in every string-typed column, including JSON stored as text. Missing values and literal strings such as `null` or `[]` are not counted as blanks.

In [21]:
text_columns = books.select_dtypes(include="string").columns
blank_counts = {}

for column in text_columns:
    text_values = books[column].astype("string")
    blank_counts[column] = int(text_values.str.strip().eq("").sum())

display(pd.Series(blank_counts, name="blank_string_rows").to_frame())

,blank_string_rows
amazon_business_money_record_number,0
amazon_isbn_10,935
amazon_isbn_13,7
amazon_raw_record_json,0
author_json,0
average_rating,0
bought_together_json,0
canonical_isbn_13,0
details_json,0
images_json,0


Objective: Find every column that contains lists or arrays, then count empty collections and show their percentages.

In [24]:

def is_collection(value):
    return isinstance(value, (list, tuple, np.ndarray))


collection_columns = [
    column for column in books.columns if books[column].map(is_collection).any()
]


def is_empty_collection(value):
    return isinstance(value, (list, tuple, np.ndarray)) and len(value) == 0


empty_counts = {}
for column in collection_columns:
    empty_counts[column] = int(books[column].map(is_empty_collection).sum())

empty_summary = pd.DataFrame.from_dict(
    empty_counts, orient="index", columns=["empty_collection_rows"]
)
empty_summary["percent_of_books"] = (
    100 * empty_summary["empty_collection_rows"] / row_count
).round(2)
display(empty_summary.sort_values("empty_collection_rows", ascending=False))

,empty_collection_rows,percent_of_books
ol_isbn_10,2422,26.81
ol_isbn_13,1233,13.65
description,1028,11.38
ol_publishers,792,8.77
features,43,0.48
ol_work_keys,2,0.02
categories,0,0.00
isbn_aliases,0,0.00
matched_isbn_aliases,0,0.00
toc_entries,0,0.00


## 4. Check identifiers and inspect repeated-key groups

Count repeated identifiers without dropping records. “Extra rows” counts occurrences beyond the first; “rows in repeated groups” includes every member. Matching ISBNs, edition keys, or titles do not by themselves establish that two rows should be merged.

Objective: Count missing, unique, and repeated identifiers to identify groups that need inspection.

In [25]:
identifier_columns = ["parent_asin", "canonical_isbn_13", "ol_edition_key"]
identifier_rows = []

for column in identifier_columns:
    identifiers = books[column].astype("string")
    missing = identifiers.isna() | identifiers.str.strip().eq("").fillna(False)
    counts = identifiers.loc[~missing].value_counts()
    repeated_groups = counts.loc[counts > 1]
    identifier_rows.append({
        "identifier": column,
        "missing_or_blank": int(missing.sum()),
        "distinct_present_values": len(counts),
        "repeated_groups": len(repeated_groups),
        "rows_in_repeated_groups": int(repeated_groups.sum()),
        "extra_rows": int((repeated_groups - 1).sum()),
    })

display(pd.DataFrame(identifier_rows).set_index("identifier"))

,missing_or_blank,distinct_present_values,repeated_groups,rows_in_repeated_groups,extra_rows
identifier,,,,,
parent_asin,0,9034,0,0,0
canonical_isbn_13,0,8995,38,77,39
ol_edition_key,0,8877,150,307,157


Objective: Show 12 rows with repeated ISBNs so we can compare their titles and identifiers.

In [26]:
inspection_columns = [
    "parent_asin", "canonical_isbn_13", "ol_edition_key", "title", "ol_title"
]
repeated_isbn = books["canonical_isbn_13"].duplicated(keep=False)

display(
    books.loc[repeated_isbn, inspection_columns]
    .sort_values(["canonical_isbn_13", "parent_asin"])
    .head(12)
)

,parent_asin,canonical_isbn_13,ol_edition_key,title,ol_title
174,0071220917,9780071388672,/books/OL18479103M,Sales Force Management,Project leadership
197,0071388672,9780071388672,/books/OL18479103M,Project Leadership,Project leadership
167,0070222924,9780071484992,/books/OL18003746M,Influencer: The Power to Change Anything,Influencer
321,007148499X,9780071484992,/books/OL18003746M,Influencer: The Power to Change Anything,Influencer
813,0131930079,9780131930070,/books/OL3401821M,"Business Ethics, A Teaching and Learning Class...",Business ethics
9019,B007C2JQS6,9780131930070,/books/OL3401821M,Business Ethics Concepts and Cases 6th Ed.,Business ethics
1099,0142000280,9780142000281,/books/OL7360123M,Getting Things Done: The Art of Stress-Free Pr...,Getting Things Done
9020,B007J5D4QQ,9780142000281,/books/OL7360123M,Getting Things Done: The Art of Stress-Free Pr...,Getting Things Done
1100,0142000981,9780142000984,/books/OL18791857M,Coal: A Human History,Coal
8978,B000LMPL5G,9780142000984,/books/OL18791857M,Coal: A Human History,Coal


These first 12 rows are examples to inspect, not a representative sample of duplicate groups. Compare titles and identifiers before deciding the recommendation unit. Related editions or works may also need to be grouped when defining evaluation splits.

## 5. Inspect numeric parsing and distributions

Create a separate numeric view for summaries. `errors="coerce"` turns unparseable values into nulls in this diagnostic view so we can count them explicitly; it is not a proposed cleaning policy. No values are imputed and no thresholds are fitted.

Objective: Check whether numeric-looking strings can be parsed, then summarize their values in a separate diagnostic view.

In [27]:
numeric_columns = [
    "average_rating", "rating_number", "raw_toc_item_count", "toc_entry_count"
]
numeric_books = books[numeric_columns].apply(pd.to_numeric, errors="coerce")
invalid_counts = (books[numeric_columns].notna() & numeric_books.isna()).sum()

display(pd.DataFrame({
    "raw_dtype": books[numeric_columns].dtypes.astype(str),
    "unparseable_rows": invalid_counts,
    "nonfinite_numeric_rows": (
        numeric_books.notna() & ~np.isfinite(numeric_books)
    ).sum(),
}))
display(numeric_books.describe(percentiles=[0.25, 0.5, 0.75, 0.99]).T)

,raw_dtype,unparseable_rows,nonfinite_numeric_rows
average_rating,str,0,0
rating_number,str,0,0
raw_toc_item_count,str,0,0
toc_entry_count,str,0,0


,count,mean,std,min,25%,50%,75%,99%,max
average_rating,9034.0,4.283894,0.503954,1.0,4.1,4.4,4.60,5.00,5.0
rating_number,9034.0,292.614678,2132.876100,1.0,9.0,26.0,84.75,4856.50,90755.0
raw_toc_item_count,9034.0,15.664490,21.073010,3.0,9.0,12.0,17.00,86.67,581.0
toc_entry_count,9034.0,15.662497,21.072910,3.0,9.0,12.0,17.00,86.67,581.0


Objective: Plot the distributions of average ratings, rating counts, and TOC entry counts.

Long tails can compress most records into the first bins. Large counts are observations to investigate, not automatic outliers to delete. These are catalog distributions; no supervised target has been defined, so they do not establish class imbalance.

## 6. Describe source coverage and categories

Source labels can reveal how the catalog was selected. The notebook does not verify the source matching process. For multi-valued categories, count each label once per parent record; percentages can sum to more than 100%.

Objective: Count source labels and show the 15 most frequent categories and their percentages of books.

In [34]:
display(books["main_category"].value_counts(dropna=False).rename("records").to_frame())
display(books["match_type"].value_counts(dropna=False).rename("records").to_frame())

category_memberships = (
    books[["parent_asin", "categories"]]
    .explode("categories")
    .dropna(subset=["categories"])
    .drop_duplicates()
)
category_counts = category_memberships["categories"].value_counts()
category_summary = category_counts.rename("books_with_label").to_frame()
category_summary["percent_of_books"] = (100 * category_counts / row_count).round(2)
display(category_summary.head(15))

,records
main_category,
Books,9032
NaN,2


,records
match_type,
exact_isbn_alias,9034


,books_with_label,percent_of_books
categories,,
Books,9034,100.00
Business & Money,9034,100.00
Management & Leadership,2374,26.28
Economics,963,10.66
Business Culture,885,9.80
Marketing & Sales,767,8.49
Job Hunting & Careers,666,7.37
Industries,645,7.14
Business Development & Entrepreneurship,535,5.92


## 7. Inspect TOC shape and count disagreements

Compare the two stored counts and the actual length of `toc_entries`. Check overlap before adding warning counts: one record can satisfy more than one condition.

Objective: Find TOC format and count differences, check overlapping cases, and display the affected records.

In [30]:
display(books["raw_toc_shape"].value_counts(dropna=False).rename("records").to_frame())

string_form_toc = books["raw_toc_shape"].eq("list[string]")
stored_count_mismatch = numeric_books["raw_toc_item_count"].ne(
    numeric_books["toc_entry_count"]
)
actual_toc_lengths = books["toc_entries"].map(len)
actual_length_mismatch = actual_toc_lengths.ne(numeric_books["toc_entry_count"])

toc_checks = pd.Series({
    "string_form_toc": int(string_form_toc.sum()),
    "stored_count_mismatch": int(stored_count_mismatch.sum()),
    "both_conditions": int((string_form_toc & stored_count_mismatch).sum()),
    "either_condition": int((string_form_toc | stored_count_mismatch).sum()),
    "actual_length_mismatch": int(actual_length_mismatch.sum()),
}, name="records")
display(toc_checks.to_frame())

toc_review_columns = [
    "parent_asin", "raw_toc_shape", "raw_toc_item_count", "toc_entry_count"
]
display(books.loc[string_form_toc | stored_count_mismatch, toc_review_columns])

,records
raw_toc_shape,
list[dict],9030
list[string],4


,records
string_form_toc,4
stored_count_mismatch,17
both_conditions,0
either_condition,21
actual_length_mismatch,0


,parent_asin,raw_toc_shape,raw_toc_item_count,toc_entry_count
201,0071396985,list[string],10,10
652,0072262362,list[dict],17,16
749,0130650757,list[dict],15,14
1186,0198770545,list[dict],12,11
1453,0275985792,list[dict],11,10
1755,037572303X,list[dict],43,42
2616,0470821108,list[dict],4,3
2985,0595366333,list[dict],17,16
3192,0717803880,list[dict],6,4
3442,0757000940,list[dict],12,11


## 8. Verify the artifact is unchanged

Objective: Compare the before and after checksums, and confirm the row count and numeric source types remain unchanged.

In [ ]:
checksum_after = file_checksum(raw_path)

print(f"Checksum before EDA: {checksum_before}")
print(f"Checksum after EDA:  {checksum_after}")
assert checksum_after == checksum_before, "The raw Parquet changed during EDA."
assert len(books) == row_count
assert all(books[column].dtype == "str" for column in numeric_columns)
print("Raw file checksum and row count unchanged; numeric source columns remain strings.")

## Stop here: define evaluation before developing transformations

This notebook ends with descriptive questions. It does not create a processed dataset.

Before using these findings to select features, thresholds, missing-value rules, or chunking strategies:

1. Define the recommendation task and what the evaluation will hold out: queries, users, books, or related works/editions.
2. Establish development and evaluation data appropriate to that task. If the goal is unseen-book generalization, keep related editions and chunks of the same work in the same group.
3. Fit learned transformations (for example, median imputation, scaling, or a learned vocabulary) using training data only.
4. Use development/validation evidence to choose strategies, then reserve final evaluation for the agreed test data.

We have inspected the full saved catalog. It is development evidence, not a newly untouched test set. A retrieval candidate catalog and held-out evaluation queries have different roles; we must define those roles before splitting anything.

No model or ranking-quality claim follows from this notebook. Record observations and unresolved questions in [docs/data.md](../docs/data.md).

Reference: [scikit-learn: avoiding data leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).